# Machine Learning: offline companion notebook
Run in a clean kernel. See SETUP.md. Read corresponding lessons and vary one parameter after each example.


## M01 — preprocessing pipeline
Dependencies: numpy scikit-learn.


In [ ]:
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
X = np.array([[20,'a'],[40,'b'],[30,'a'],[50,'b'],[np.nan,'a'],[60,'b']], dtype=object)
pre = ColumnTransformer([('num',make_pipeline(SimpleImputer(strategy='median'),StandardScaler()),[0]),('cat',OneHotEncoder(handle_unknown='ignore'),[1])])
model = make_pipeline(pre,LogisticRegression()).fit(X,[0,1,0,1,0,1])
assert model.predict([[None,'new']]).shape == (1,)
print(model.predict_proba([[35,'new']]))

## M02 — supervised baselines
Dependencies: scikit-learn.


In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import balanced_accuracy_score
X,y = load_breast_cancer(return_X_y=True)
Xdev,Xte,ydev,yte = train_test_split(X,y,test_size=.2,random_state=7,stratify=y)
Xtr,Xval,ytr,yval = train_test_split(Xdev,ydev,test_size=.25,random_state=8,stratify=ydev)
models=[DummyClassifier(),make_pipeline(StandardScaler(),LogisticRegression(max_iter=1000)),RandomForestClassifier(n_estimators=50,random_state=7)]
scores=[]
for model in models:
    model.fit(Xtr,ytr)
    scores.append(balanced_accuracy_score(yval,model.predict(Xval)))
    print(type(model).__name__,'validation',scores[-1])
best=models[max(range(len(scores)),key=scores.__getitem__)]
best.fit(Xdev,ydev)
print('Selected model test score',balanced_accuracy_score(yte,best.predict(Xte)))


## M03 — cross validation
Dependencies: scikit-learn.


In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score
X,y = load_breast_cancer(return_X_y=True)
Xtr,Xte,ytr,yte = train_test_split(X,y,stratify=y,test_size=.25,random_state=2)
search = RandomizedSearchCV(RandomForestClassifier(n_estimators=30,random_state=2),{'max_depth':[2,4,None],'min_samples_leaf':[1,3,5]},n_iter=3,cv=StratifiedKFold(3,shuffle=True,random_state=2),scoring='average_precision',random_state=2)
search.fit(Xtr,ytr)
print(search.best_params_,average_precision_score(yte,search.predict_proba(Xte)[:,1]))

## M04 — clustering
Dependencies: scikit-learn.


In [ ]:
from sklearn.datasets import make_moons
from sklearn.cluster import DBSCAN,KMeans
from sklearn.metrics import silhouette_score
X,_ = make_moons(n_samples=300,noise=.04,random_state=7)
for model in [KMeans(n_clusters=2,n_init=10,random_state=7),DBSCAN(eps=.18,min_samples=5)]:
    labels = model.fit_predict(X)
    mask = labels != -1
    if len(set(labels[mask])) > 1: print(type(model).__name__,silhouette_score(X[mask],labels[mask]),'coverage',mask.mean())
# Inspect geometry too; silhouette can favor convex clusters.

## M05 — forecast backtest
Dependencies: numpy.


In [ ]:
import numpy as np
rng = np.random.default_rng(7)
y = 20+np.sin(np.arange(100)*2*np.pi/7)+rng.normal(0,.2,100)
for cutoff in (50,65,80):
    history = y[:cutoff]
    actual = y[cutoff:cutoff+7]
    baseline = history[-7:]
    assert len(actual)==len(baseline)
    print(cutoff, np.mean(abs(actual-baseline)))